# Marker-zeroing and center-inclusive radius controls

**Role:** Analysis.

Reproduce the original marker-zeroing intervention on any restored GIN or FiLM model. The center fate remains unchanged by default; optional hop zero is a separate center-identity intervention. Zeroing an exclusive fate makes the source unassigned, so these are reference-dependent prediction contrasts, not cell removal or unknown-fate effects. Compare with replacement/masking in the companion fate-interventions notebook.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/models/gnn.py").is_file())  # Locate the repository from the notebook’s working directory.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
ROOT = PROJECT_ROOT  # Repository root used to resolve data and result paths.

## Size figure definitions

These study-specific definitions build the summaries or panels displayed below. Cohort selection, marker labels, thresholds and figure layouts remain visible here; shared numerical and plotting primitives are imported from `src/`. Defining these functions does not run inference or load results.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import NullFormatter
import numpy as np
import pandas as pd

SIZE_MODEL_LABELS = {
    "gin_head_size": "GIN: N in head",  # Head-only model with log cell count.
    "gin_film_size": "GIN: N in FiLM and head",  # FiLM model with log cell count.
    "gin_head_size_geometry": "GIN: N + geometry in head",  # Head-only control with log count and geometry.
    "gin_film_size_geometry": "GIN: N in FiLM; N + geometry in head",  # FiLM control with log count and geometry.
}  # Legend labels for the compared size-conditioned models.
SIZE_CURVE_STYLES = {
    "observed": dict(
        color="#2878b5",  # Display color for color.
        linestyle="-",  # Matplotlib line style.
        marker="o",  # Matplotlib point symbol.
        label="Observed size",  # Legend text for this curve.
    ),  # Style for curves grouped by observed organoid count.
    "sweep": dict(
        color="#d45a31",  # Display color for color.
        linestyle="--",  # Matplotlib line style.
        marker="s",  # Matplotlib point symbol.
        label="Sweep size",  # Legend text for this curve.
    ),  # Style for fixed-neighborhood supplied-count curves.
}  # Display styles for observed and swept counts.

def size_pair_rows(table, center, source, hop):
    return table[(table.center_marker == center) &
                 (table.source_marker_name == source) & (table.hop == hop)]

def size_draw_overlay(ax, panel, *, min_organoids, show_confidence, x_scale):
    ax.axhline(0, color="0.65", linewidth=0.65, zorder=0)
    available = False
    for kind, style in SIZE_CURVE_STYLES.items():
        group = panel[panel.analysis == kind].sort_values("n")
        valid = group.n_organoids >= min_organoids
        # Keep masked rows to leave gaps where support is inadequate.
        ax.plot(group.n, group["mean"].where(valid), linewidth=1.6,
                markersize=3.5, **style)
        if show_confidence:
            ax.fill_between(group.n, group.ci_low.where(valid), group.ci_high.where(valid),
                            color=style["color"], alpha=0.13, linewidth=0)
        available = available or bool(valid.any())
    if not available:
        ax.text(0.5, 0.5, "Insufficient support", transform=ax.transAxes,
                ha="center", va="center", color="0.5", fontsize=8)
    ax.set_xscale(x_scale)
    if x_scale == "log":
        counts = panel.n.dropna()
        if len(counts):
            ticks = np.unique(np.rint(np.geomspace(counts.min(), counts.max(), 3)).astype(int))
            ax.set_xticks(ticks, labels=[str(n) for n in ticks])
        ax.xaxis.set_minor_formatter(NullFormatter())
    ax.ticklabel_format(axis="y", style="sci", scilimits=(-2, 2))
    ax.tick_params(labelsize=8)
    ax.spines[["top", "right"]].set_visible(False)

def size_match_limits(ax, pair, min_organoids, show_confidence):
    valid = pair[pair.n_organoids >= min_organoids]
    columns = ["mean", "ci_low", "ci_high"] if show_confidence else ["mean"]
    values = valid[columns].to_numpy().reshape(-1)
    values = values[np.isfinite(values)]
    if len(values):
        low, high = min(0., float(values.min())), max(0., float(values.max()))
        margin = max(high - low, 1e-8) * 0.08
        ax.set_ylim(low - margin, high + margin)

def size_legend(fig):
    handles = [Line2D([], [], linewidth=1.6, markersize=4, **style)
               for style in SIZE_CURVE_STYLES.values()]
    fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, 0.965),
               ncol=2, frameon=False)

def size_plot_size_overlay_grid(
    table, model, hop, *, min_organoids=5, show_confidence=True,
    x_scale="log", include_unmarked=True, match_model_limits=True,
    ylabel="Δ curvature",
):
    """One center-by-source grid for a model, with both analyses on each axis."""
    if x_scale not in ("log", "linear"):
        raise ValueError("x_scale must be 'log' or 'linear'.")
    if model not in set(table.model) or hop not in set(table.hop):
        raise ValueError("Requested model or hop is absent from the saved tables.")
    sources = sorted(table.source_marker_name.unique())
    centers = [marker for marker in sources if marker in set(table.center_marker)]
    if include_unmarked and "unmarked" in set(table.center_marker):
        centers.append("unmarked")
    fig, axes = plt.subplots(len(centers), len(sources), squeeze=False, sharex=True,
                             figsize=(3.0 * len(sources), 2.45 * len(centers)))
    for i, center in enumerate(centers):
        for j, source in enumerate(sources):
            ax = axes[i, j]
            pair = size_pair_rows(table, center, source, hop)
            size_draw_overlay(ax, pair[pair.model == model], min_organoids=min_organoids,
                          show_confidence=show_confidence, x_scale=x_scale)
            if match_model_limits:
                size_match_limits(ax, pair, min_organoids, show_confidence)
            if i == 0:
                ax.set_title(f"Ablate {source}", fontsize=10)
            if j == 0:
                ax.set_ylabel(f"Center: {center}\n{ylabel}", fontsize=9)
            if i == len(centers) - 1:
                ax.set_xlabel("Cell count N", fontsize=9)
    fig.suptitle(f"{SIZE_MODEL_LABELS.get(model, model)} — exact hop {hop}\n"
                 "Observed-size and fixed-neighborhood sweep effects", fontsize=15, y=0.998)
    size_legend(fig)
    fig.tight_layout(rect=(0, 0, 1, 0.94), h_pad=1.5)
    return fig

def size_plot_geometric_normalization_comparison(table, pair, models, *, min_organoids=5):
    """Raw and dimensionless overlays side by side, with one row per model."""
    center, source, hop = pair
    fig, axes = plt.subplots(len(models), 2, squeeze=False, sharex=True, sharey='col',
                             figsize=(11, 3.5 * len(models)))
    for row, model in enumerate(models):
        for col, (metric, label) in enumerate([
            ('delta_mu', 'Δ curvature (original units)'),
            ('delta_relative', 'ΔK / Kref(N) (dimensionless)'),
        ]):
            panel = size_pair_rows(table, center, source, hop)
            panel = panel[(panel.model == model) & (panel.metric == metric)]
            size_draw_overlay(axes[row, col], panel, min_organoids=min_organoids,
                          show_confidence=True, x_scale='log')
            axes[row, col].set_ylabel(label)
            axes[row, col].set_title(SIZE_MODEL_LABELS.get(model, model), fontsize=11)
            if row == len(models) - 1:
                axes[row, col].set_xlabel('Cell count N')
    fig.suptitle(f'Ablate {source} near {center}, exact hop {hop}\n'
                 'Raw versus geometric normalization', y=0.995)
    size_legend(fig)
    fig.tight_layout(rect=(0, 0, 1, 0.9))
    return fig

## Select model and sampling settings

Use saved validation organoids. Exact-hop perturbations and optional center perturbations share the same sampled recipients; the extracted receptive field must cover both the model depth and requested ablation radius.

In [ ]:
import json, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
from src.artifacts.runs import AnalysisRun
from src.analysis.metrics.evaluation import prediction_table
from src.artifacts.paths import resolve_training_run
TRAINING_NOTEBOOK = 'gin_depth_training'  # Folder under training_results containing the saved run.
TRAINING_RUN = None  # Set a run directory name; None selects only if exactly one exists.
RUN_DIR = resolve_training_run(ROOT, TRAINING_NOTEBOOK, TRAINING_RUN)  # Also accepts an explicit historical path.
run = AnalysisRun(RUN_DIR)
display(run.records)

# Saved model selection
MODEL_KEY = run.records.iloc[0].key  # Checkpoint key selected from the saved model catalog.

# Inference resources
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'  # Use CUDA when available, otherwise CPU.
selection = run.select(MODEL_KEY, device=DEVICE)
model = selection['model']
marker_names = selection['marker_names']
g_train, g_val = selection['groups']['train'], selection['groups']['val']
OUTPUT_DIR = RUN_DIR/'analysis'/'marker_zeroing'/MODEL_KEY  # Directory for this analysis’s tables and figures.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MAX_CENTERS_PER_ORGANOID = 5  # Maximum sampled recipient centers per organoid.
ABLATION_RADIUS = 2  # Maximum exact-hop distance to perturb.
INCLUDE_CENTER = False  # Also edit hop 0; this changes recipient identity.
MODE = 'single'  # single edits one positive source; all edits the whole ring.
SEED = 1701  # Random seed for reproducible sampling or initialization.

## Construct recipients and evaluate edits

Sample recipients without using curvature or prediction error. Extract sufficient neighborhoods while preserving the original organoid globals. Each case is paired with its own intact prediction.

In [ ]:
from src.data.subgraphs import build_ego_subgraphs_for_graph
from src.analysis.interventions.perturbation import compute_perturbation_influence_maps
rng=np.random.default_rng(SEED)
subgraphs=[]
for i,g in enumerate(g_val):
    subgraphs.extend(build_ego_subgraphs_for_graph(g,num_hops=max(ABLATION_RADIUS,getattr(model,'num_layers',0)),
        max_centers=MAX_CENTERS_PER_ORGANOID,rng=rng,graph_idx=i))
result=compute_perturbation_influence_maps(subgraphs,model,marker_names,k_hops=ABLATION_RADIUS,
    mode=MODE,max_subgraphs=None,device=DEVICE,target_transform=selection['transform'],
    include_center=INCLUDE_CENTER,return_case_effects=True,normalize_by='cases')
cases=pd.DataFrame(result['case_effects'])
cases.to_csv(OUTPUT_DIR/'zeroing_cases.csv.gz',index=False)
display(cases.head())

## Center/source effects and support

Rows are unchanged recipient marker identities and columns are removed source markers. Effects are edited minus intact prediction in physical curvature units. These descriptive case averages are accompanied by support; overlapping recipients are not independent replicates.

In [ ]:
values=np.asarray(result['delta_mu_cmarker'])
fig,axes=plt.subplots(1,len(result['hops']),figsize=(6*len(result['hops']),5),squeeze=False)
for i,hop in enumerate(result['hops']):
    ax=axes[0,i];im=ax.imshow(values[i],cmap='coolwarm')
    ax.set(title=f'Exact hop {hop}',xticks=range(len(marker_names)),xticklabels=marker_names,
           yticks=range(len(marker_names)),yticklabels=marker_names,xlabel='Source',ylabel='Recipient')
    ax.tick_params(axis='x',rotation=90);fig.colorbar(im,ax=ax,label='Edited − intact curvature')
plt.tight_layout();plt.show()
display(pd.DataFrame({'hop':result['hops'],'cases':np.asarray(result['cases_cmarker']).sum(axis=(1,2))}))

## Optional: geometric normalization of saved size sweeps

Inspect the existing head-only/FiLM size-sweep case tables and their fold-specific geometric references without retraining. This optional section uses an explicit historical run, independent of the checkpoint selected above.

In [ ]:
SHOW_SAVED_NORMALIZATION = False  # Load and plot the historical geometric normalization.

<a id="geometric-normalization"></a>

### Post-training geometric normalization (no retraining or ablation inference)

Run **only this section**, starting with its imports below, to reproduce these results in
a fresh kernel. It reads the saved case tables, split membership, and per-organoid JSON
metadata. No graphs, checkpoints, training, or model predictions are loaded or executed.
The displayed analysis uses the completed run with the sphericity filter enabled
(`SPHERICITY_MAX=0.94`) and the two models **without geometry inputs to the head**.

For each training fold, fit one observation per organoid by ordinary least squares:

$$\log \widehat A_f(N)=\alpha_f+\beta_f\log N,\qquad
K_{\rm ref,f}(N)=4\pi/\widehat A_f(N),\qquad
E=\Delta K/K_{\rm ref,f}(N).$$

This is a positive geometric-mean area reference, not an arithmetic-mean area prediction.
The calibration uses training organoids only. Held-out organoids diagnose calibration;
they do not fit it. Both models use the same fold reference. Observed cases use observed N;
sweeps use supplied N. The model inputs, organoid baseline, and predictions stay unchanged.
A positive effect of 0.2 means an increase equal to 20% of the reference sphere's curvature.
Area and Gaussian curvature must use compatible length units for the result to be dimensionless.

Normalization occurs before averaging cases/seeds within an organoid, followed by equal
organoid weighting. The 95% intervals bootstrap organoids with reference fits and models
held fixed; they do not include calibration or retraining uncertainty. Raw effects are
recomputed from the same cases; means and support are checked against the saved summaries.
Bootstrap draws can differ from the four-model analysis because only two models are summarized.

The reference removes a specified geometric size trend; it does not establish that growth
is geometrically similar. In particular, bending introduced by division is not new biological
evidence. The original observed/sweep difference in neighborhood composition remains.

In [ ]:
if SHOW_SAVED_NORMALIZATION:
    from pathlib import Path
    import json
    import sys
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from IPython.display import display

    GEOM_PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src').is_dir())  # Locate the repository from the notebook’s working directory.
    if str(GEOM_PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(GEOM_PROJECT_ROOT))
    from src.analysis.normalization.geometry import load_normalized_effects
    # Study-specific functions are defined in this notebook.

    GEOM_RUN_NAME = None  # Name of the new run or the explicitly selected saved run.
    if GEOM_RUN_NAME is not None:
        GEOM_RUN_DIR = GEOM_PROJECT_ROOT / 'results_experiments/size_conditioned_ablation' / GEOM_RUN_NAME  # Training run to create or saved run to analyze.
    else:
        GEOM_RUN_DIR = GEOM_PROJECT_ROOT / 'results_experiments/size_conditioned_ablation/20260910_164825_all_depth2_four_models_baseline_with_size'  # Training run to create or saved run to analyze.

    GEOM_SETTINGS = json.loads((GEOM_RUN_DIR / 'settings.json').read_text())  # Explicit run configuration, saved alongside the models.

    # Dataset and targets
    GEOM_DATA_DIR = GEOM_PROJECT_ROOT / 'training_data' / GEOM_SETTINGS['DATASET_NAME']  # Dataset directory supplying graphs and metadata.
    GEOM_MODELS = ['gin_head_size', 'gin_film_size']  # Saved model families included in these plots.

    # Ablation sampling
    GEOM_HOPS = [1, 2]  # Exact graph-distance rings to analyze.

    # Statistical summaries
    GEOM_MIN_ORGANOIDS = 5  # Minimum distinct organoids required for reported effects.
    GEOM_BOOTSTRAP_SAMPLES = 1000  # Number of organoid bootstrap resamples.
    GEOM_BOOTSTRAP_SEED = GEOM_SETTINGS['SPLIT_SEED']  # Random seed for the bootstrap intervals.

    # Display options
    GEOM_FOCUS_PAIRS = [('LGR5', 'LGR5', 1), ('LGR5', 'Lysozyme', 1), ('AldoB', 'AldoB', 1)]  # Recipient/source/hop combinations shown in detail.

    # Paths and outputs
    GEOM_OUTPUT_DIR = GEOM_RUN_DIR / 'geometric_normalization'  # Directory for this analysis’s tables and figures.
    GEOM_OUTPUT_DIR.mkdir(exist_ok=True)
    print('Source run:', GEOM_RUN_DIR.name)
    print('Sphericity filter:', GEOM_SETTINGS['APPLY_SPHERICITY_FILTER'], 'threshold:', GEOM_SETTINGS['SPHERICITY_MAX'])
    print('Models:', GEOM_MODELS)

    def geom_show_save(fig, name):
        for extension in ['png', 'pdf']:
            fig.savefig(GEOM_OUTPUT_DIR / f'{name}.{extension}', dpi=160, bbox_inches='tight')
        display(fig)
        plt.close(fig)


In [ ]:
if SHOW_SAVED_NORMALIZATION:
    geom_results = load_normalized_effects(
        GEOM_RUN_DIR, GEOM_DATA_DIR, GEOM_MODELS,
        bootstrap_samples=GEOM_BOOTSTRAP_SAMPLES, seed=GEOM_BOOTSTRAP_SEED, progress=print,
    )
    geom_summary = geom_results['summary']
    # Verify the reconstructed raw effects use exactly the original cases/weighting/binning.
    for geom_mode, geom_coord in [('observed', 'size_bin'), ('sweep', 'evaluated_n')]:
        geom_saved = pd.read_csv(GEOM_RUN_DIR / f'tables/{geom_mode}_delta_mu_summary.csv')
        geom_saved = geom_saved[geom_saved.model.isin(GEOM_MODELS)].rename(columns={geom_coord: 'coordinate'})
        geom_raw = geom_summary[(geom_summary.analysis == geom_mode) & (geom_summary.metric == 'delta_mu')]
        geom_keys = ['model', 'center_marker', 'source_marker_name', 'hop', 'coordinate']
        geom_check = geom_saved.merge(geom_raw, on=geom_keys, suffixes=('_saved', '_new'), validate='one_to_one')
        assert len(geom_check) == len(geom_saved) == len(geom_raw)
        for geom_column in ['mean', 'n_organoids', 'n_rows']:
            np.testing.assert_allclose(geom_check[geom_column + '_saved'], geom_check[geom_column + '_new'],
                                       rtol=1e-9, atol=1e-12, equal_nan=True)
    print('Raw effect means and support match the original saved summaries; bootstrap intervals were recomputed.')
    for geom_name, geom_table in geom_results.items():
        geom_table.to_csv(GEOM_OUTPUT_DIR / f'{geom_name}.csv', index=False)
    (GEOM_OUTPUT_DIR / 'settings.json').write_text(json.dumps(dict(
        source_run=str(GEOM_RUN_DIR), metadata_dir=str(GEOM_DATA_DIR), models=GEOM_MODELS,
        reference='OLS log(area) on log(N), training organoids only, no smearing correction',
        bootstrap_samples=GEOM_BOOTSTRAP_SAMPLES, bootstrap_seed=GEOM_BOOTSTRAP_SEED,
        min_organoids=GEOM_MIN_ORGANOIDS, focus_pairs=GEOM_FOCUS_PAIRS,
    ), indent=2))
    display(geom_results['references'])


#### Check the geometric reference

The fitted exponent beta describes the empirical area–N scaling in the filtered cohort.
The left panel shows the training-fold reference curves against observed area. The middle
panel shows residuals on held-out organoids (each organoid once for folded validation).
A systematic trend would question the power-law reference. The right panel shows the
curvature scale used as the denominator; geometry is not being added to either GNN.

In [ ]:
if SHOW_SAVED_NORMALIZATION:
    geom_fig, geom_axes = plt.subplots(1, 3, figsize=(16, 4.2))
    geom_cohort = geom_results['cohort']
    geom_n = np.geomspace(geom_cohort.n_cells.min(), geom_cohort.n_cells.max(), 150)
    geom_axes[0].scatter(geom_cohort.n_cells, geom_cohort.surface_area, s=8, alpha=0.2, color='0.4')
    for geom_fit in geom_results['references'].itertuples():
        geom_a = np.exp(geom_fit.alpha) * geom_n ** geom_fit.beta
        geom_axes[0].plot(geom_n, geom_a, label=f'Fold {geom_fit.fold}: beta={geom_fit.beta:.3f}')
        geom_axes[2].plot(geom_n, 4 * np.pi / geom_a)
    geom_axes[0].set(xscale='log', yscale='log', xlabel='Cell count N', ylabel='Surface area', title='Training-fold geometric references')
    geom_axes[0].legend(fontsize=8)
    geom_val = geom_results['diagnostics'].query("role == 'val'").copy()
    geom_axes[1].scatter(geom_val.n_cells, geom_val.log_area_residual, s=8, alpha=0.2, color='0.4')
    geom_val['n_bin'] = pd.qcut(geom_val.n_cells, 8, duplicates='drop')
    geom_residual_bins = geom_val.groupby('n_bin', observed=True).agg(n=('n_cells', 'median'), residual=('log_area_residual', 'median'))
    geom_axes[1].plot(geom_residual_bins.n, geom_residual_bins.residual, 'o-', color='black', label='Bin median')
    geom_axes[1].axhline(0, color='gray', lw=1)
    geom_axes[1].set(xscale='log', xlabel='Observed N (held out)', ylabel='log(A) − log(A reference)', title='Held-out reference residuals')
    geom_axes[1].legend()
    geom_axes[2].set(xscale='log', yscale='log', xlabel='Cell count N', ylabel='Kref(N) = 4π / A reference(N)', title='Positive normalization denominator')
    geom_fig.tight_layout()
    geom_show_save(geom_fig, 'area_reference_diagnostics')


#### Raw and normalized effects side by side

Each figure shows the same marker pair in original curvature units (left) and relative to
the geometric reference (right). Rows distinguish head-only and FiLM models. Blue circles
show observed size; orange squares show fixed-neighborhood sweeps. Axes match between models
within each column. The endpoint table reports changes between the smallest and largest
supplied N; these are descriptive differences of the plotted means, not significance tests.

In [ ]:
if SHOW_SAVED_NORMALIZATION:
    for geom_pair in GEOM_FOCUS_PAIRS:
        geom_fig = size_plot_geometric_normalization_comparison(
            geom_summary, geom_pair, GEOM_MODELS, min_organoids=GEOM_MIN_ORGANOIDS)
        geom_show_save(geom_fig, f'{geom_pair[0]}_{geom_pair[1]}_hop{geom_pair[2]}_raw_vs_normalized')
    geom_endpoint_rows = []
    for (geom_model, geom_center, geom_source, geom_hop, geom_metric), geom_group in geom_summary.query("analysis == 'sweep'").groupby(
            ['model', 'center_marker', 'source_marker_name', 'hop', 'metric']):
        if (geom_center, geom_source, geom_hop) not in GEOM_FOCUS_PAIRS:
            continue
        geom_group = geom_group.sort_values('n')
        geom_low, geom_high = geom_group.iloc[0], geom_group.iloc[-1]
        geom_endpoint_rows.append(dict(model=geom_model, center=geom_center, ablated=geom_source, hop=geom_hop,
            metric=geom_metric, n_low=geom_low.n, n_high=geom_high.n,
            low_effect=geom_low['mean'], high_effect=geom_high['mean'],
            endpoint_change=geom_high['mean'] - geom_low['mean'],
            n_organoids_low=geom_low.n_organoids, n_organoids_high=geom_high.n_organoids))
    geom_endpoints = pd.DataFrame(geom_endpoint_rows)
    geom_endpoints.to_csv(GEOM_OUTPUT_DIR / 'selected_pair_endpoints.csv', index=False)
    display(geom_endpoints)


#### Normalized effects for all marker pairs

Each panel overlays observed-size and sweep-size effects, now in dimensionless units.
Y limits match between the two models for the same marker pair and hop. Points with fewer
than the configured minimum number of organoids are masked. The full raw and normalized
summaries, organoid-level means, fitted reference parameters, calibration metadata, and
PNG/PDF figures are saved in the run's `geometric_normalization` directory.

In [ ]:
if SHOW_SAVED_NORMALIZATION:
    geom_normalized = geom_summary[geom_summary.metric == 'delta_relative'].copy()
    for geom_model in GEOM_MODELS:
        for geom_hop in GEOM_HOPS:
            geom_fig = size_plot_size_overlay_grid(
                geom_normalized, geom_model, geom_hop, min_organoids=GEOM_MIN_ORGANOIDS,
                ylabel='ΔK / Kref(N)', match_model_limits=True,
            )
            geom_show_save(geom_fig, f'{geom_model}_hop{geom_hop}_normalized_overlays')
    print('Saved post-training normalization to:', GEOM_OUTPUT_DIR)


#### Reading the displayed normalization results

These outputs use 1,100 organoids from the completed filtered run. The five training-fold
area exponents range from **0.495 to 0.529**, so the reference curvature falls approximately
as N to the power −0.51. Held-out log-area R² ranges from **0.52 to 0.70**. The residual
plot shows scatter and a mild size trend: this is an empirical reference, not an exact
physical scaling law. Across the 165–791 sweep, the normalization multiplier grows by
approximately 2.2× within a fold.

For LGR5 ablation near LGR5 at hop 1, the FiLM raw endpoint effect changes from −0.00277
to −0.00161, while the dimensionless effect changes from −0.0367 to −0.0471. Its
intermediate-size turning behavior remains visible. The head-only normalized effect
changes from −0.0159 to −0.0494. Thus conclusions about endpoint magnitude depend on
whether effects are measured in absolute curvature or relative to the size reference.

Lysozyme ablation near LGR5 becomes more negative across the sweep in both units for both
models. The AldoB–AldoB sweep crosses zero in both units. A positive normalizer preserves
the sign of every individual effect, although fold-specific scaling can reweight an
aggregated curve. These are descriptive model sensitivities, not tests of causal biology.